In [ ]:
# pip install libraries

!pip install sentence-transformers faiss-cpu langchain
!pip install pypdf langchain_community
!pip install langchain-text-splitters

In [ ]:
# upload the document
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
  print("Uploaded:", filename)

In [ ]:
# document ingestion pipeline

from langchain_community.document_loaders import PyPDFLoader

pdf_file = list(uploaded.keys())[0]
loader = PyPDFLoader(pdf_file)
documents = loader.load()
print("Number of pages loaded", len(documents))

In [ ]:
print(documents[0])

In [ ]:
# chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)

chunks = text_splitter.split_documents(documents)

print("total chunk created:", len(chunks))

In [ ]:
print(chunks[0].page_content)

In [ ]:
print(chunks[1].page_content)

In [ ]:
# generate embedding from the chunk

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode([chunk.page_content for chunk in chunks])
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Total vectors in ths index", index.ntotal)



In [ ]:
print(dimension)

In [ ]:
import matplotlib.pyplot as plt

def visualize_embedding(vector, chunk_text = None, max_text_chars = 500):
    vector = np.array(vector)

    v_min = vector.min()
    v_max = vector.max()

    # 3. Min-Max Normalization: Squishes the numbers between 0 and 1
    # This maps the "raw math" to "colors" that a human can see.
    normalized = (vector - v_min) / (v_max - v_min)

    # 4. Reshape to (1, N) so matplotlib sees it as a single row (a band)
    color_band = normalized.reshape(1, -1)

    if chunk_text is not None:
        print("--- Chunk Text ---")
        print(chunk_text[:max_text_chars])
        print("-" * 18)

    # 5. Plotting the "Heatmap" of the vector
    plt.figure(figsize=(12, 2))
    # 'viridis' goes from purple (low) to yellow (high)
    plt.imshow(color_band, aspect="auto", cmap="viridis")
    plt.yticks([]) # Hide the y-axis because there is only 1 row
    plt.xlabel(f"Embedding Dimensions ({len(vector)})")
    plt.title("Vector Visual Representation (Heatmap)")
    plt.colorbar(label="Normalized Value") # Useful to see the scale
    plt.show()


In [ ]:
visualize_embedding(embeddings[5],chunks[5].page_content)

In [ ]:
# retrieval

query = input("Enter your question: ")
query_embedding = model.encode([query])

visualize_embedding(query_embedding, query)

In [ ]:
D, I = index.search(np.array(query_embedding), k=5)
retrieved_chunks = [chunks[i] for i in I[0]]

print("Top retrieved chunks: \n")
for chunk in retrieved_chunks:
  print(chunk.page_content)
  print("Page:", chunk.metadata["page"])
  print("------------------------------")


In [ ]:
# reranking

from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLm-L6-v2")
pairs = [(query, chunk.page_content) for chunk in retrieved_chunks]
scores = reranker.predict(pairs)

ranked_chunks = sorted(zip(scores, retrieved_chunks),
                       key = lambda x: x[0],
                       reverse = True
                       )
print("After reranking: \n")
for score, chunk in ranked_chunks:
  print("scores:", score)
  print(chunk.page_content)
  print("Page number: ", chunk.metadata["page"])
  print("-----------------------------")

  # 34, 19, 20, 27, 36


In [ ]:
# combine reranked chunk with context

top_chunks = [chunk.page_content for _,chunk in ranked_chunks[:3]]
context = "\n\n".join(top_chunks)
print(context)

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google.genai import Client
from google.colab import userdata
import numpy as np

# 1. Initialize the Client
client = Client(api_key=userdata.get('GOOGLE_API_KEY'))

def ask_bot():
    print("--- 🤖 RAG Chatbot Started! Type 'exit' to stop. ---")

    while True:
        # Get user input
        query = input("\n👤 You: ")

        if query.lower() in ['exit', 'quit', 'bye']:
            print("👋 Goodbye!")
            break

        # 2. RETRIEVAL STEP
        # We turn your question into a vector using the same model we used for chunks
        query_vector = model.encode([query])

        # Search the FAISS index for the top 3 most relevant chunks
        # D is distance, I is the index/ID of the chunk
        D, I = index.search(np.array(query_vector), k=3)

        # Pull the actual text from your 'chunks' list using the IDs found by FAISS
        retrieved_chunks = [chunks[i].page_content for i in I[0]]
        context = "\n\n".join(retrieved_chunks)

        # 3. GENERATION STEP (Gemini)
        prompt = f"""
        Answer the question using the context below.
        Context:
        {context}

        Question:
        {query}

        Provide a concise answer.
        If you don't find the info in the context, do not guess.
        Just say that the info is not found in the document.
        """

        try:
            # Note: Using gemini-1.5-flash as it's the stable production standard
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )

            print(f"\n🤖 Bot: {response.text}")

        except Exception as e:
            print(f"❌ Error: {e}")

# Run the chatbot
ask_bot()